# E5: Gated Fusion + Pretrained Emoji Embeddings

**Experiment**: Learned gate modulates emoji contribution before concatenation with text.

**Architecture**: Frozen BERT (768-d) + frozen pretrained emoji (32-d) → mean-pool emoji → gate g=σ(W·[text;emoji]+b) → concat(text, g*emoji) (800-d) → MLP → 3 classes

**Gate interpretation**: g ≈ 1 means model relies on emoji signal; g ≈ 0 means model ignores it.
Gate values are interpretability signals, not causal explanations.

**Controlled-experiment contract**:
- Same train/validation/test splits as E0–E4
- Text branch receives ONLY `text_without_emoji`
- Emoji branch receives ONLY `emoji_list`
- Emoji embeddings are PRETRAINED on TweetEval and FROZEN

**Prerequisite**: E2 must have been run first (provides the pretrained emoji embedding artifact).

**Runtime**: GPU (T4 recommended)

## 1. Setup

In [ ]:
# Clone repository
!rm -rf /content/SentimentAnalysis
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git /content/SentimentAnalysis
%cd /content/SentimentAnalysis
!pwd

In [ ]:
# Verify GPU
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!python -c "import torch, transformers, pandas, sklearn; print('Environment OK')"

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Build Canonical Data Pipeline

In [ ]:
%cd /content/SentimentAnalysis

!python -m src.data.inspect_datasets
!python -m src.data.stocktwits_adapter
!python -m src.data.build_final_dataset
!python -m src.data.preprocessing

In [ ]:
# Verify canonical data
import os
import pandas as pd

base = 'data/processed/canonical'
for split in ['train', 'validation', 'test']:
    path = f'{base}/final_{split}.jsonl'
    if os.path.exists(path):
        df = pd.read_json(path, lines=True)
        print(f'{split}: {len(df)} rows')
    else:
        print(f'{split}: MISSING!')

assert os.path.exists(f'{base}/final_train.jsonl'), 'Canonical data not found!'

## 3. Load Pretrained Emoji Embedding from E2

In [ ]:
import os
import shutil

drive_src = '/content/drive/MyDrive/SentimentAnalysis/models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt'
local_dst = 'models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt'

os.makedirs('models/emoji_embeddings', exist_ok=True)

if os.path.exists(drive_src):
    shutil.copy2(drive_src, local_dst)
    print('Copied pretrained embedding from Drive')
else:
    print(f'ERROR: Pretrained embedding not found at {drive_src}')
    print('You must run E2 first and backup results to Drive.')
    raise FileNotFoundError(f'Missing: {drive_src}')

In [ ]:
# Verify the embedding artifact
import torch

artifact = torch.load(local_dst, map_location='cpu')
print(f"Embedding shape: {artifact['emoji_embedding'].shape}")
print(f"Vocab size: {artifact['vocab_size']}")

assert artifact['emoji_embedding'].shape == (1610, 32)
print('Pretrained emoji embedding verified.')

## 4. Train E5

In [ ]:
%cd /content/SentimentAnalysis

# Run E5 training + evaluation + gate analysis
!RUN_E5=1 python run_e5.py

## 5. Check Results

In [ ]:
!find results/E5 -maxdepth 2 -type f | sort

In [ ]:
import json

with open('results/E5/metrics.json') as f:
    metrics = json.load(f)

print('=' * 60)
print('E5 OFFICIAL RESULTS')
print('=' * 60)
print(f"Accuracy:        {metrics['accuracy']:.6f}")
print(f"Macro Precision: {metrics['macro_precision']:.6f}")
print(f"Macro Recall:    {metrics['macro_recall']:.6f}")
print(f"Macro F1:        {metrics['macro_f1']:.6f}")
print(f"Best Epoch:      {metrics['best_epoch']}")
print(f"Best Val F1:     {metrics['best_val_macro_f1']:.6f}")

In [ ]:
# View gate analysis (E5-specific)

with open('results/E5/gate_analysis.json') as f:
    gate = json.load(f)

print('=' * 60)
print('E5 GATE ANALYSIS')
print('=' * 60)
print(f"Overall gate mean:   {gate['overall_mean']:.4f}")
print(f"Overall gate std:    {gate['overall_std']:.4f}")
print(f"Overall gate median: {gate['overall_median']:.4f}")
print()
for cls in ['Bearish', 'Neutral', 'Bullish']:
    print(f"{cls}: mean={gate[f'{cls}_mean']:.4f} std={gate[f'{cls}_std']:.4f} median={gate[f'{cls}_median']:.4f}")

In [ ]:
from IPython.display import Image, display

display(Image('results/E5/confusion_matrix.png'))
display(Image('results/E5/training_history.png'))

## 6. Backup to Google Drive

In [ ]:
import os

drive_dir = '/content/drive/MyDrive/SentimentAnalysis/results/E5'
os.makedirs(drive_dir, exist_ok=True)

!cp -r results/E5/* {drive_dir}/

print('E5 results backed up to Drive.')
!find {drive_dir} -maxdepth 2 -type f | sort